In [8]:
# toy model specifications
n_genes = 10
n_metabolites = 15
n_reactions = 20
n_exchange_reactions = 5
n_gprs = 15
single_gpr_rate = 0.8 # lower = more complex/less real GPRs, higher= more single-gene GPRs

# simulartion specifications
n_media_sims = 2
max_gene_kos = 2


In [9]:
import cobra as cb
import numpy as np
import random

def generate_random_gpr(genes, max_depth=3, single_gpr_rate=0.6):
    """Recursively builds a valid GPR string."""
    if max_depth == 0 or random.random() < single_gpr_rate:
        return random.choice(genes)
    
    op = random.choice(["and", "or"])
    left = generate_random_gpr(genes, max_depth - 1, single_gpr_rate)
    right = generate_random_gpr(genes, max_depth - 1, single_gpr_rate)
    
    return f"({left} {op} {right})"

def create_random_toy_model(id="ToyModel", n_metabolites=20, n_reactions=30, n_genes=5, 
                            reversible_fraction=0.3, n_exchanges=5, n_gprs=10,
                            single_gpr_rate=0.6):
    model = cb.Model(id)
    
    # 1. Setup Genes and GPR pool
    genes = [f"G{i}" for i in range(n_genes)]
    unused_genes = set(genes)
    
    # 2. Create Metabolites (Cytosolic and Extracellular)
    mets_c = [cb.Metabolite(f"met_{i}_c", compartment="c") for i in range(n_metabolites)]
    
    # Select indices for extracellular metabolites
    exchange_indices = random.sample(range(n_metabolites), min(n_exchanges, n_metabolites))
    # Map index to extracellular metabolite object
    mets_e_map = {i: cb.Metabolite(f"met_{i}_e", compartment="e") for i in exchange_indices}
    
    model.add_metabolites(mets_c)
    model.add_metabolites(list(mets_e_map.values()))
    
    # 3. Create Internal Reactions
    for i in range(n_reactions):
        rxn = cb.Reaction(f"R_{i}")
        
        # Ensure flow towards the end of the metabolite list
        s_idx = random.sample(range(0, n_metabolites - 1), random.randint(1, 2))
        p_idx = random.sample(range(max(s_idx) + 1, n_metabolites), random.randint(1, 1))
        
        met_dict = {mets_c[s]: -1.0 for s in s_idx}
        met_dict.update({mets_c[p]: 1.0 for p in p_idx})
        
        rxn.add_metabolites(met_dict)
        rxn.lower_bound = -1000.0 if random.random() < reversible_fraction else 0.0
        rxn.upper_bound = 1000.0
        model.add_reactions([rxn])

    # 4. Add Exchange and Transport Reactions
    for i in exchange_indices:
        # Exchange: Boundary <-> Extracellular
        ex_rxn = model.add_boundary(mets_e_map[i], type="exchange", lb=-10.0, ub=1000.0)
        ex_rxn.id = f"EX_met_{i}"
        
        # Transport: Extracellular <-> Cytosolic
        trans_rxn = cb.Reaction(f"T_met_{i}")
        trans_rxn.add_metabolites({mets_e_map[i]: -1.0, mets_c[i]: 1.0})
        trans_rxn.lower_bound = -1000.0 if random.random() < reversible_fraction else 0.0
        trans_rxn.upper_bound = 1000.0
        model.add_reactions([trans_rxn])

    # We target R_ and T_ reactions, excluding EX_ and the eventual biomass
    eligible_rxns = [r for r in model.reactions if not r.id.startswith("EX_")]
    random.shuffle(eligible_rxns)

    # Phase 1: Assign n_gprs complex rules
    for i in range(min(n_gprs, len(eligible_rxns))):
        eligible_rxns[i].gene_reaction_rule = generate_random_gpr(genes, single_gpr_rate=single_gpr_rate)
    
    # Phase 2: Identify "orphaned" genes not in any GPR
    used_genes = set()
    for rxn in model.reactions:
        if rxn.gene_reaction_rule:
            # Extract gene IDs from the rule string (simple approach for toy models)
            for g in genes:
                if g in rxn.gene_reaction_rule:
                    used_genes.add(g)
    
    orphaned_genes = [g for g in genes if g not in used_genes]

    # Phase 3: Assign orphaned genes as singular GPRs to reactions with NO GPR
    empty_rxns = [r for r in eligible_rxns if not r.gene_reaction_rule]
    
    for i, gene in enumerate(orphaned_genes):
        if i < len(empty_rxns):
            empty_rxns[i].gene_reaction_rule = gene
        else:
            # Optional: if we run out of empty reactions, 
            # we could append it to an existing GPR using 'or'
            print(f"Warning: No empty reactions left for orphaned gene {gene}")

    # 6. Add Biomass Objective
    biomass = cb.Reaction("bio1")
    # Consumes the final two metabolites
    biomass.add_metabolites({mets_c[-1]: -1.0, mets_c[-2]: -1.0})
    biomass.lower_bound = 0
    biomass.upper_bound = 1000
    model.add_reactions([biomass])
    model.objective = biomass
    
    return model

# --- Execution ---
toy_model = create_random_toy_model(
    n_metabolites=n_metabolites, 
    n_reactions=n_reactions, 
    n_genes=n_genes,
    n_exchanges=n_exchange_reactions,
    n_gprs=n_gprs,
    single_gpr_rate=single_gpr_rate
)

print(f"Model: {toy_model.id} | Reactions: {len(toy_model.reactions)}")
for rxn in toy_model.reactions: # Print first 10 for brevity
    print(f"{rxn.id:10} | GPR: {rxn.gene_reaction_rule:35} | Eq: {rxn.reaction}")

solution = toy_model.optimize()
print(f"\nOptimal Growth Rate: {solution.objective_value:.4f}")

Model: ToyModel | Reactions: 31
R_0        | GPR:                                     | Eq: met_2_c <=> met_6_c
R_1        | GPR: G3                                  | Eq: met_4_c + met_5_c --> met_10_c
R_2        | GPR: G2                                  | Eq: met_10_c + met_5_c --> met_14_c
R_3        | GPR: G4 or G2                            | Eq: met_12_c <=> met_14_c
R_4        | GPR: G8 or G0                            | Eq: met_5_c --> met_7_c
R_5        | GPR: (G8 or (G8 and G1)) and G4          | Eq: met_12_c + met_5_c <=> met_14_c
R_6        | GPR:                                     | Eq: met_2_c <=> met_8_c
R_7        | GPR: (G4 and G0) or G6                   | Eq: met_6_c --> met_8_c
R_8        | GPR: G2                                  | Eq: met_3_c + met_9_c --> met_12_c
R_9        | GPR: G7                                  | Eq: met_2_c --> met_9_c
R_10       | GPR: G0                                  | Eq: met_11_c --> met_13_c
R_11       | GPR:                     

# create Gene Knockout Data

In [10]:
n_exs = len([r for r in toy_model.reactions if r.id.startswith("EX_")])

# simulate different media conditions
media_vals = np.random.uniform(-10, 0, size=(n_exs, n_media_sims))
media_mask = np.random.rand(n_exs, n_media_sims) < 0.6
media_vals[~media_mask] = 0.0
cutoff = -0.5
media_vals[media_vals > cutoff] = 0.0
media_vals.shape

# each media condition has a random subset of genes knocked out
n_gene_sims = min(n_genes, max_gene_kos)  # Ensure we don't simulate more genes KOs than exist
gene_knockouts = np.zeros(shape=(n_gene_sims), dtype=int)
genes_to_ko = np.random.choice(n_genes, size=(n_gene_sims,), replace=False)
for i, g in enumerate(genes_to_ko):
    gene_knockouts[i] = g

gene_knockouts
# # simulate growth for each condition
growth_rates = np.zeros(shape=(n_media_sims, n_gene_sims))
for i in range(n_media_sims):
    for j in range(n_gene_sims):
        sim_model = toy_model.copy()
        
        # Apply media condition
        for ex_idx, ex_rxn in enumerate([r for r in sim_model.reactions if r.id.startswith("EX_")]):
            ex_rxn.lower_bound = media_vals[ex_idx, i]
        
        # Apply gene knockouts
        sim_model.genes.get_by_id(f"G{gene_knockouts[j]}").knock_out()

        
        # Optimize and store growth rate
        solution = sim_model.optimize()
        growth_rates[i, j] = solution.objective_value if solution.status == 'optimal' else 0.0

# store results in DF
import pandas as pd
gene_cols = [f"G{g}" for g in range(n_genes)]
ex_cols = [r.id for r in toy_model.reactions if r.id.startswith("EX_")]

columns = ex_cols + gene_cols + ["GrowthRate"]
df = pd.DataFrame(columns=columns, index=[f"Sim_{i}" for i in range(n_media_sims * n_gene_sims)])
for i in range(n_media_sims):
    for j in range(n_gene_sims):
        row_idx = i * n_gene_sims + j
        ex_values = media_vals[:, i]
        gene_values = [1 if g == genes_to_ko[j] else 0 for g in range(n_genes)]
        df.loc[f"Sim_{row_idx}", ex_cols] = ex_values
        df.loc[f"Sim_{row_idx}", gene_cols] = gene_values
        df.loc[f"Sim_{row_idx}", "GrowthRate"] = growth_rates[i, j]

df

,EX_met_3,EX_met_9,EX_met_13,EX_met_14,EX_met_12,G0,G1,G2,G3,G4,G5,G6,G7,G8,G9,GrowthRate
Sim_0,0.0,-2.91795,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0.0
Sim_1,0.0,-2.91795,0.0,0.0,0.0,0,0,0,0,0,1,0,0,0,0,0.0
Sim_2,0.0,-1.849896,0.0,-6.356893,-2.750087,0,0,0,0,0,0,0,0,0,1,9.106979
Sim_3,0.0,-1.849896,0.0,-6.356893,-2.750087,0,0,0,0,0,1,0,0,0,0,5.478438


In [11]:
print(gene_knockouts)

[9 5]
